In [46]:
# # from abaqus import mdb
# # from abaqusConstants import *
import os
import json
import sys
# # from MESH.meshAlt import *
# # from BCONDITIONS.casing import *
# # from BCONDITIONS.conditions import *
# # from GEOMETRY.geometries import *
# # from JOBS.job import *
from JSONS.ImportTools import *
# # from MATERIALS.materials import *
# # from GEOMETRY.sets import *
# # from GEOMETRY.assembly import *
# from GEOMETRY_PS.geometry_PS import *

path_project = r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL'

if path_project not in sys.path:
    sys.path.append(path_project)

with open(r'C:\Users\leticia\Documents\GitHub\Abaqus_WELL\wellbore_closure_planestrain.json') as f:
    data = json.load(f)
    print(f"Data keys: {data.keys()}")

# Variables read from json (geometry) #####################################

# name_phase = '3dda7930-6dbf-4d05-87f2-d2809a3e9fc6'
# name_tubular = 'LIN_09_875'
if "Phases" not in data["AnalysisData"]:
    print("Chave 'Phases' não encontrada")
    print(data["AnalysisData"].keys())

name_phase = data["AnalysisData"]["Phases"]
print(name_phase)
phase_data = data["Phases"][name_phase]
if phase_data:
    name_tubular = phase_data["Casing"][0]["Tubular"]
else:
    print(f"Phase '{name_phase}' not found in data['Phases']")

############ Rock dimensions ############################
diameter_wellbore = phase_data["HoleDiameter"]
outer_radius_wellbore = diameter_wellbore / 2
outer_radius_wellbore = outer_radius_wellbore * 0.0254  # Convert from inches to meters
thickness_wellbore = outer_radius_wellbore * 0.9  # Variavel da espessura da rocha
inner_radius_wellbore = outer_radius_wellbore - thickness_wellbore

########### Casing / Pipe dimensions ####################
outer_diameter_pipe = data["Tubulars"][name_tubular]['OD']
outer_radius_pipe = (outer_diameter_pipe / 2) * 0.1 # 10% do valor do raio externo para criar um espaço entre a parede do tubo e a borda do modelo
outer_radius_pipe = outer_radius_pipe * 0.0254  # Convert from inches to meters
thickness_pipe = data["Tubulars"][name_tubular]['Thickness']
thickness_pipe = thickness_pipe * 0.1 * 0.0254  # 10% do valor da espessira (inches to meters)
inner_radius_pipe = outer_radius_pipe - thickness_pipe
stand_off = data["AnalysisData"]["StandOff"] / 100   # Convert from inches to meters
min_wall_thickness = data["Tubulars"][name_tubular]["MinThickness"]
# min_wall_thickness = min_wall_thickness / 100  # Convert from inches to meters
# min_wall_thickness = (1 - min_wall_thickness)   # Convert from inches to meters
# thickness_min = thickness_pipe * min_wall_thickness
# ovality = data["Tubulars"][name_tubular]["Ovality"] / 100

########## Annulus dimensions ###########################
outer_radius_annular = inner_radius_wellbore
inner_radius_annular = outer_radius_pipe
thickness_annular = outer_radius_annular - inner_radius_annular

l_depth = data["AnalysisData"]["Depth"]
print(f"The bottom of the wellbore is at: {-l_depth} meters")

lithology = data["Lithology"]
for layer in lithology:
        if l_depth >= layer["Top"] and l_depth < layer["Bottom"]:
            layer_rock = layer["Rock"]
            print(f"Layer at depth {l_depth} meters: {layer_rock}")

Data keys: dict_keys(['AnalysisData', 'ThermalGradient', 'Tubulars', 'Lithology', 'InSituStresses', 'Rocks', 'Cements', 'SteelGrades', 'Phases', 'Events', 'Fluids'])
812c492a-c945-4184-be51-841c5fb86b15
The bottom of the wellbore is at: -2600.0 meters
Layer at depth 2600.0 meters: SANDSTONE


In [50]:
examples = {}

# casing_type = "VM110"
casing_type = data["Tubulars"][name_tubular]["Material"]
print(f"Casing type: {casing_type}") 
# Seleciona o tipo de aço para o casing definido no json (ex: VM-95) e pega as propriedades do material a partir do json
steelgrade_info = data["SteelGrades"][casing_type]
# Seleciona o Gradiente Geotérmico definido no "AnalysisData"
data_geothermal = data["AnalysisData"]["GeothermalGradient"]
print(f"Selected Geothermal Gradient: {data_geothermal}")
# Seleciona o Gradiente Térmico presente e definido antes
thermalGradient = data["ThermalGradient"][data_geothermal]
print(f"Selected Thermal Gradient: {thermalGradient}")
# Retorna uma lista de todos os fluidos com "ThermalGradient" = "data_geothermal"

name_fluido = next(
    (name for name, info in data["Fluids"].items() 
    if info.get("ThermalGradient") == data_geothermal), None)
print(f"Selected Fluid: {name_fluido}")

examples["STEEL"] = {
    "behavior": data["SteelGrades"][casing_type]["Law"],
    'density': data["SteelGrades"][casing_type]["ElasticParameters"]["Density"],
    'elastic': (data["SteelGrades"][casing_type]["ElasticParameters"]["Young"]*1e9,
                data["SteelGrades"][casing_type]["ElasticParameters"]["Poisson"]),
    'conductivity': data["SteelGrades"][casing_type]["ThermalParameters"]["Conductivity"],
    'specific_heat': data["SteelGrades"][casing_type]["ThermalParameters"]["SpecificHeat"],
    'expansion': data["SteelGrades"][casing_type]["ThermalParameters"]["ThermalExpansion"],
    "type": "Casing"
}

examples["FLUID"] = {
    "behavior": "ELASTIC",
    'density': data["Fluids"][name_fluido]["Density"],
    'compressibility': data["Fluids"][name_fluido]["Compressibility"],
    'ThermalExpansion': data["Fluids"][name_fluido]["ThermalExpansion"],
    "type": "Fluid"
}

examples[layer_rock] = {
"behavior": data["Rocks"][layer_rock]["Law"],
'density': data["Rocks"][layer_rock]["ElasticParameters"]["Density"],
'elastic': (data["Rocks"][layer_rock]["ElasticParameters"]["Young"]*1e9,
            data["Rocks"][layer_rock]["ElasticParameters"]["Poisson"]),
'conductivity': data["Rocks"][layer_rock]["ThermalParameters"]["Conductivity"],
'specific_heat': data["Rocks"][layer_rock]["ThermalParameters"]["SpecificHeat"],
'expansion': data["Rocks"][layer_rock]["ThermalParameters"]["ThermalExpansion"],
"type": "Rock"
}

if "MohrCoulombParameters" in data["Rocks"][layer_rock]:
    mc = data["Rocks"][layer_rock]["MohrCoulombParameters"]
    examples[layer_rock].update({
    'friction_angle': mc["FrictionAngle"],
    'dilatancy_angle': mc["DilatancyAngle"],
    'cohesion': mc["Cohesion"],
    "lab_data": ((20001698.76, 0.0), )
    })

if "DoublePowerParameters" in data["Rocks"][layer_rock]:
        examples[layer_rock]["DoublePowerParameters"] = data["Rocks"][layer_rock]["DoublePowerParameters"]

material_examples = {
    "PIPE": {
        "partName": "PIPE",
        "sectionName": 'STEEL_Section',
        "isSolid": True
    },
    "FLUID": {
        "partName": "FLUID",
        "sectionName": 'FLUID_Section',
        "isSolid": True
    }
}

Casing type: K55
Selected Geothermal Gradient: temp. drilling 14in - 60 degC
Selected Thermal Gradient: [{'Depth': 2000.0, 'Temperature': 15.0}, {'Depth': 4550.0, 'Temperature': 60.0}]
Selected Fluid: drilling_14in (60 deg)


In [1]:
# -*- coding: utf-8 -*-
from abaqusConstants import *
from abaqus import mdb
import numpy as np
import section
import regionToolset
import displayGroupMdbToolset as dgm
import part
import sys
# import os

from caeModules import *
from driverUtils import executeOnCaeStartup
executeOnCaeStartup()

class PlaneStrainPart:

    def __init__(self, name, data, span="half"):
        self.name = name
        self.data = data
        self.span = span
        self.get_geometry()

    def get_geometry(self):
        center1 = self.data.get("center1", [0,0])
        center2 = self.data.get("center2", [0,0])
        Ro1 = self.data.get("outer_radius")
        Ro2 = self.data.get("outer_radius")
        Ri1 = Ro1 - self.data.get("thickness")
        Ri2 = Ro2 - self.data.get("thickness")
        self.geometry = {
            "center1": center1,
            "center2": center2,
            "Ro1": Ro1,
            "Ro2": Ro2,
            "Ri1": Ri1,
            "Ri2": Ri2
        }

    def create_part(self, modelName):
        m = mdb.models[modelName]
        sketch_name = '__profile__' + self.name
        s = m.ConstrainedSketch(name=sketch_name, sheetSize=200.0)
        s.setPrimaryObject(option=STANDALONE)

        g1 = s.ArcByCenterEnds(center=self.geometry["center1"], point1 = (self.geometry["Ro1"], 0.0), point2 = (self.geometry["Ro2"], 0.0), direction=COUNTERCLOCKWISE)
        g2 = s.ArcByCenterEnds(center=self.geometry["center2"], point1 = (self.geometry["Ri1"], 0.0), point2 = (self.geometry["Ri2"], 0.0), direction=COUNTERCLOCKWISE)

        if self.span == "half":
            g3 = s.Line(point1=(-self.geometry["Ro1"], 0.0), point2=(self.geometry["Ro1"], 0.0))
            s.autoTrimCurve(curve1=g1, point1=(self.geometry["center1"][0], -self.geometry["Ro2"]))
            s.autoTrimCurve(curve1=g2, point1=(self.geometry["center2"][0], -self.geometry["Ri2"]))
            s.autoTrimCurve(curve1=g3, point1=self.geometry["center1"])
        elif self.span == "quarter":
            g3 = s.Line(point1=(self.geometry["center1"][0], 0.0), point2=(self.geometry["Ro1"], 0.0))
            g4 = s.Line(point1=(self.geometry["center1"][0], 0.0), point2=(0.0, self.geometry["Ro2"]))
            s.autoTrimCurve(curve1=g1, point1=(self.geometry["center1"][0], -self.geometry["Ro2"]))
            s.autoTrimCurve(curve1=g2, point1=(self.geometry["center2"][0], -self.geometry["Ri2"]))
            s.autoTrimCurve(curve1=g3, point1=self.geometry["center1"])
            s.autoTrimCurve(curve1=g4, point1=self.geometry["center1"])
        p = m.Part(name=self.name, dimensionality=TWO_D_PLANAR, type=DEFORMABLE_BODY)
        p.BaseShell(sketch=s)
        s.unsetPrimaryObject()
        del m.sketches[sketch_name]
        return p

    def set_Mesh():
        pass
        # self.elemTypes = (CPE4, CPE3)
        # elemType1 = mesh.ElemType(elemCode=self.elemTypes[0], elemLibrary=STANDARD)
        # elemType2 = mesh.ElemType(elemCode=self.elemTypes[1], elemLibrary=STANDARD)

        # p.setMeshControls(regions=p.faces, technique=STRUCTURED)
        # p.setElementType(regions=(p.faces,), elemTypes=(elemType1, elemType2))

        # ec = geomAuxiliar.filterEdges(p.edges, geomAuxiliar.isCircularEdge)
        # eh = geomAuxiliar.filterEdges(p.edges, geomAuxiliar.isHorizontalEdge)

        # p.seedEdgeByNumber(edges=eh, number=self.numElems[0], constraint=FINER)
        # p.seedEdgeByNumber(edges=ec, number=self.numElems[1], constraint=FINER)
        # p.generateMesh()

    def add_to_assembly(self, modelName):
        m = mdb.models[modelName]
        m.rootAssembly.Instance(name=self.name,
                                part=m.parts[self.name],
                                dependent=ON)
        m.rootAssembly.regenerate()

    def create_spec_sets(self, modelName):
        pass

    def create_contact_sets(self):
        pass
             
    def create_sets(self, modelName):
        m = mdb.models[modelName]
        p = m.parts[self.name]
        f = p.faces
        
######### Criando os Sets inteiros para facilitar a atribuição de seções e condições de contorno ##########
        # p.Set(name='ALL', faces=p.faces)
        all_faces = f[0:len(f)]
        p.Set(faces=all_faces, name='FASEI_' + self.name.upper())
        
        ##### FASEI_REV_TT    ##################
        tol = 0.001
        all_coords = [v.pointOn[0][1] for v in p.vertices]
        min_y_global = min(all_coords)
        base_edges = p.edges.getByBoundingBox(
            xMin = -1e20, yMin=min_y_global - tol, zMin=-tol,
            xMax = 1e20, yMax=min_y_global + tol, zMax=tol
        )
        p.Set(edges=base_edges, name='FASEI_' + self.name.upper() + '_TT')
########## FASEI_REV_OD e FASEI_REV_ID ###########################
        if self.span == "half":
            angles = (0.25*np.pi, 0.75*np.pi)
        elif self.span == "quarter":
            angles = (0.25*np.pi,)
        else:
            angles = (0.25*np.pi, 0.75*np.pi, 1.25*np.pi, 1.75*np.pi)
        od_edges = p.edges[:0]
        id_edges = p.edges[:0]
        for angle in angles:
            xy1 = (self.geometry["Ro1"]*np.cos(angle) + self.geometry["center1"][0],
                   self.geometry["Ro2"]*np.sin(angle) + self.geometry["center1"][1],
                   0.0)
            xy2 = (self.geometry["Ri1"]*np.cos(angle) + self.geometry["center2"][0],
                   self.geometry["Ri2"]*np.sin(angle) + self.geometry["center2"][1],
                   0.0)
            od_edges += p.edges.findAt((xy1,))
            id_edges += p.edges.findAt((xy2,))
        p.Set(edges = od_edges,name='FASEI_' + self.name.upper() + '_OD')
        p.Set(edges = id_edges,name='FASEI_' + self.name.upper() + '_ID')
                    
if __name__ == "__main__":
    mdb.models.changeKey(fromName='Model-1', toName='MyFirstModel')
    AnnulusPart = PlaneStrainPart("AnnulusPart1",
                     data={"center1": [0,0],
                           "center2": [0,0],
                           "outer_radius": 10,
                           "thickness": 2},)
    AnnulusPart.create_part("MyFirstModel")
    AnnulusPart.create_sets("MyFirstModel")
    AnnulusPart.add_to_assembly("MyFirstModel")
    print("Annulus created and added to assembly.")

ModuleNotFoundError: No module named 'abaqusConstants'